# Signal Idea 1: Hidden Markov Market Regime Detection

Research notebook capturing one implementable signal concept from quant literature (non-HFT, not price-only).


In [ ]:
# Setup: imports and robust paths
import sys
from pathlib import Path

import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES_ROOT = [CWD, CWD.parent]

for root in CANDIDATES_ROOT:
    if (root / "config" / "settings.py").exists():
        PROJECT_ROOT = root
        break
else:
    PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def resolve_first_existing(paths: list[Path], fallback: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    return fallback

DATA_DIR = resolve_first_existing(
    [root / "data" / "factors" for root in CANDIDATES_ROOT],
    PROJECT_ROOT / "data" / "factors",
)
DUCKDB_PATH = resolve_first_existing(
    [root / "data" / "factors" / "factors.duckdb" for root in CANDIDATES_ROOT],
    DATA_DIR / "factors.duckdb",
)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"DuckDB path  : {DUCKDB_PATH}")
print(f"Data directory exists: {DATA_DIR.exists()}")
print(f"DuckDB exists: {DUCKDB_PATH.exists()}")
if not DUCKDB_PATH.exists():
    print("⚠️ DuckDB file missing. Use parquet files directly for prototype tests.")


In [ ]:
from IPython.display import display
# Quick local data inventory for feasibility
candidates = [
    DATA_DIR / "prices.parquet",
    DATA_DIR / "factors_price.parquet",
    DATA_DIR / "factors_all.parquet",
    DATA_DIR / "macro.parquet",
    DATA_DIR / "macro_z.parquet",
]

rows = []
for p in candidates:
    rows.append({
        "file": p.name,
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / (1024**2), 2) if p.exists() else None,
    })

display_df = pd.DataFrame(rows)
display_df


## What this idea is
Use a Hidden Markov Model (HMM) to infer latent market regimes (risk-on, neutral, risk-off) from a **joint feature set** rather than price alone.

Suggested state inputs (daily):
- Cross-sectional dispersion of returns
- Market beta spread (high-beta minus low-beta basket)
- Volatility factor (`vol_60d`) aggregate
- Macro z-scores from `macro_z.parquet`

The signal is regime probability (for example, P(risk-on)) used to scale gross exposure or tilt factor sleeves.


## Paper lineage
- Hamilton (1989), *A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle*.
- Ang and Bekaert (2002), *International Asset Allocation with Regime Shifts*.

Why relevant: regime-switching models are established in macro-finance and map directly to portfolio risk-budgeting.


## Feasibility in this project
**Feasibility: High**

- We already have prices + factors + macro parquet artifacts.
- MVP implementation can use `hmmlearn` with 2-3 latent states and monthly re-estimation.
- No HFT dependency; this is daily/weekly horizon.

Implementation steps:
1. Build daily feature matrix from existing parquet files.
2. Fit HMM on expanding window.
3. Convert filtered state probabilities into exposure multipliers.
4. Backtest with existing portfolio engine.


## Data requirements and availability
- **Already local**: prices, factor time series, macro z-scores.
- **Optional external**: VIX from Stooq/Yahoo if we want richer stress features.

If missing, easiest sources:
- FRED (free API) for macro rates/spreads.
- Stooq/Yahoo for VIX and broad index proxies.

Data is realistic to obtain with low operational overhead.


## Minimal prototype notebook cell
```python
# pseudo-outline
# 1) Load parquet features
# 2) Standardize train window only
# 3) Fit 2-3 state Gaussian HMM
# 4) Produce state probabilities and map to exposure
# 5) Evaluate sharpe, drawdown, turnover
```


## Recommended next experiment in this repo
1. Build a minimal feature table using currently available parquet files.
2. Run a simple walk-forward backtest using existing portfolio metrics.
3. Add one external dataset only if it materially improves explanatory power.
